# 📜 API Contract đề xuất — Frontend & Security Tests (TV4 → TV2)

> **Trạng thái: ĐỀ XUẤT, chưa chốt.** TV2 sở hữu API. Notebook này ghi lại những gì Frontend (`frontend/`)
> và bộ test bảo mật (`tests/security/`) đang giả định.

**Cách đọc notebook này**
1. Phần **A** (chữ): quy ước chung, mã lỗi, danh sách endpoint.
2. Phần **B** (code chạy được): bật mock API rồi gọi thử từng endpoint, để thấy tận mắt API trả gì khi bị tấn công chéo tenant.
3. Phần **C**: các điểm TV2 cần xác nhận.

Khi TV2 chốt Swagger/OpenAPI, chỉ cần sửa **2 file**:

| Phía | File |
|---|---|
| Frontend | `frontend/src/api/client.ts` |
| Security tests | `tests/security/api_paths.py` |

---
## A. Quy ước chung

| Quy ước | Giá trị đề xuất | Vì sao |
|---|---|---|
| Base path | `/api/v1` | Có version, đổi API không làm hỏng client cũ |
| Tenant trong URL | `/api/v1/orgs/{tenantId}/...` | `TenantMembershipInterceptor` đọc `tenantId` từ path rồi kiểm tra bảng `memberships` **mỗi request** |
| Xác thực | `Authorization: Bearer <JWT Entra External ID>` | JWT chỉ chứng minh *bạn là ai*; **quyền theo tenant luôn đọc từ DB** (đây là lý do Test 2 pass) |
| Request ID | Client gửi `X-Request-Id`; API luôn trả lại `X-Request-Id` | Truy vết log theo `request_id` |
| ID | UUID | Khó đoán, nhưng **không** được coi là biện pháp bảo mật |

### Mã lỗi

| Mã | Khi nào |
|---|---|
| `401` | Thiếu, sai hoặc hết hạn JWT |
| `403` | JWT hợp lệ nhưng **không phải thành viên** `tenantId` trong path, hoặc role không đủ (Guest ghi, Member quản lý thành viên) |
| `404` | Tài nguyên không tồn tại **trong tenant này**, kể cả khi nó có thật ở tenant khác (không tiết lộ sự tồn tại) |
| `409` | Trùng `UNIQUE(tenant_id, name)` của project; hoặc tải export chưa xong |
| `400`/`422` | Dữ liệu đầu vào sai |

Body lỗi đề xuất (không stack trace, không dữ liệu tenant khác):
```json
{"error": "FORBIDDEN", "message": "...", "requestId": "..."}
```

## A1. Danh tính & Thành viên

`role` ∈ `OWNER` · `MEMBER` · `GUEST`

| Method | Path | Ai được gọi | Body → Response |
|---|---|---|---|
| GET | `/api/v1/me` | đã đăng nhập | → `{userId, email, displayName, memberships:[{tenantId, tenantName, role}]}` |
| GET | `/orgs/{tenantId}/members` | mọi role | → `[{userId, email, displayName, role, joinedAt}]` |
| POST | `/orgs/{tenantId}/members` | OWNER | `{email, role}` → `201 {userId, email, role}` (mời / thêm lại) |
| PATCH | `/orgs/{tenantId}/members/{userId}` | OWNER | `{role}` → `200` |
| DELETE | `/orgs/{tenantId}/members/{userId}` | OWNER | → `204`, có hiệu lực **ngay** ở request tiếp theo |

## A2. Project & Task

`status` ∈ `TODO` · `IN_PROGRESS` · `DONE`

| Method | Path | Ai được gọi | Ghi chú |
|---|---|---|---|
| GET | `/orgs/{tenantId}/projects` | mọi role | Cache key phía server: `tenant:{tenantId}:projects` |
| POST | `/orgs/{tenantId}/projects` | OWNER, MEMBER | `{name, description?}` → `201`; trùng tên trong cùng tenant → `409` |
| GET / PATCH / DELETE | `/orgs/{tenantId}/projects/{projectId}` | GET: mọi role · ghi: OWNER, MEMBER | DELETE → `204` |
| GET | `/orgs/{tenantId}/projects/{projectId}/tasks` | mọi role | |
| POST | `/orgs/{tenantId}/projects/{projectId}/tasks` | OWNER, MEMBER | `{title, description?, status?, assigneeUserId?}` |
| GET / PATCH / DELETE | `/orgs/{tenantId}/tasks/{taskId}` | GET: mọi role · ghi: OWNER, MEMBER | PATCH: `{title?, description?, status?, assigneeUserId?}` |

Task object:
```json
{"id", "tenantId", "projectId", "title", "description", "status", "assigneeUserId", "createdAt", "updatedAt"}
```

## A3. Audit & Export

| Method | Path | Ai được gọi | Response |
|---|---|---|---|
| GET | `/orgs/{tenantId}/audit-events?limit=50` | mọi role | `[{id, actorUserId, action, entityType, entityId, requestId, createdAt, details}]` (mới nhất trước) |
| POST | `/orgs/{tenantId}/exports` | OWNER, MEMBER | `{projectId?}` → `202 {jobId, status:"QUEUED"}` |
| GET | `/orgs/{tenantId}/exports` | mọi role | `[{jobId, status, requestedBy, createdAt, completedAt, errorCode}]` |
| GET | `/orgs/{tenantId}/exports/{jobId}` | mọi role | 1 job |
| GET | `/orgs/{tenantId}/exports/{jobId}/download` | OWNER, MEMBER | `{url, expiresAt}`: SAS **read-only, ≤ 5 phút**, chỉ khi `COMPLETED` (nếu chưa xong thì `409`) |

Export `status` ∈ `QUEUED` · `PROCESSING` · `COMPLETED` · `FAILED` · `REJECTED`

> ⚠️ **Lệch với TV3:** `prototypes/tv3-worker/.../ExportWorker.java` dùng `PENDING` (thay cho `QUEUED`) và không có `FAILED`.
> Frontend hiện chấp nhận cả hai (coi mọi trạng thái chưa kết thúc là đang chạy). TV2 + TV3 cần chốt một bộ tên.

```text
UI ──POST /exports──► API ──ghi export_jobs + gửi message──► Azure Queue ──► Worker
UI ◄─poll GET /exports/{jobId}─ API                         Worker ──CSV──► Blob private
UI ──GET .../download──► API ──User Delegation SAS (5 phút)──► UI tải thẳng từ Blob
```

⚠️ Frontend **không bao giờ** nhận storage account key, không lưu SAS URL vào localStorage, không log SAS URL.

---
## B. Thực hành: gọi thử contract trên mock API

Mock API (`tests/mock_api/server.py`) cài đặt **đúng contract ở trên**, chỉ dùng Python standard library.
Nó **không phải** backend thật; mục đích là để mọi người hiểu hành vi mong đợi trước khi TV2 hoàn thành backend.

Token giả lập có dạng `mock-<org>-<role>`, ví dụ `mock-a-owner`, `mock-b-owner`, `mock-a-guest`.

▶️ Chạy lần lượt các cell bên dưới (`Shift + Enter`). Cần `pip install requests`.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists() or (p / "infra").is_dir())
print("Repo root:", ROOT.name)

Repo root: multi-tenant-project-management


In [2]:
# Bật mock API ở một cổng trống (tự tắt ở cell cuối)
import atexit, json, socket, subprocess, sys, time
import requests

sys.path.insert(0, str(ROOT / "tests" / "security"))
from evidence import redact_obj  # che SAS/JWT trước khi in ra

with socket.socket() as s:
    s.bind(("127.0.0.1", 0)); PORT = s.getsockname()[1]
server = subprocess.Popen([sys.executable, str(ROOT / "tests" / "mock_api" / "server.py"), "--port", str(PORT)],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
atexit.register(server.terminate)
BASE = f"http://127.0.0.1:{PORT}"
for _ in range(50):
    try:
        requests.get(BASE + "/health", timeout=1); break
    except requests.ConnectionError:
        time.sleep(0.1)
print("Mock API đang chạy:", requests.get(BASE + "/health").json())


def call(token, method, path, body=None, show=True):
    """Gọi API bằng token của 1 user, in status + body rút gọn (đã che thông tin mật)."""
    h = {"Authorization": f"Bearer {token}"} if token else {}
    r = requests.request(method, BASE + "/api/v1" + path, json=body, headers=h)
    data = r.json() if r.content else None
    if show:
        text = json.dumps(redact_obj(data), ensure_ascii=False)
        print(f"{method:6} {path[:70]:70} → {r.status_code}  {text[:160]}{'…' if len(text) > 160 else ''}")
    return data

Mock API đang chạy: {'status': 'UP', 'mock': True, 'vulns': []}


### B1. Danh tính: `/me` trả danh sách org từ bảng memberships

In [3]:
me_a = call("mock-a-owner", "GET", "/me")
me_b = call("mock-b-owner", "GET", "/me", show=False)
ORG_A = me_a["memberships"][0]["tenantId"]
ORG_B = me_b["memberships"][0]["tenantId"]
print("\nOrg A =", ORG_A, "\nOrg B =", ORG_B)

GET    /me                                                                    → 200  {"userId": "4beb33ba-c817-538b-99af-f14bf6a41cdf", "email": "owner@org-a.example.com", "displayName": "Owner (A)", "memberships": [{"tenantId": "839467b6-9957-5…

Org A = 839467b6-9957-525f-9673-cf9a193b7635 
Org B = 0246d1c9-6b70-581e-a854-8b67cbc9a65f


### B2. Lấy ID thật của Org B (bằng token hợp lệ của B)

In [4]:
proj_b = call("mock-b-owner", "GET", f"/orgs/{ORG_B}/projects")[0]
task_b = call("mock-b-owner", "GET", f"/orgs/{ORG_B}/projects/{proj_b['id']}/tasks", show=False)[0]
print("Task của Org B:", task_b["id"], "-", task_b["title"])

GET    /orgs/0246d1c9-6b70-581e-a854-8b67cbc9a65f/projects                    → 200  [{"id": "5114b778-25c0-5ca8-b23c-796a3d6df27c", "tenantId": "0246d1c9-6b70-581e-a854-8b67cbc9a65f", "name": "Project 1 of B", "description": "Seed project 1", "…
Task của Org B: d397b784-3f53-533a-9efc-9d1ad195d235 - [B] Task 1


### B3. 🔒 Owner Org A tấn công Org B: phải bị chặn

| Kiểu tấn công | Ví dụ | Kỳ vọng |
|---|---|---|
| **tenant-swap** (đổi tenant trong path) | `/orgs/{B}/tasks/{taskB}` | `403`: không phải thành viên Org B |
| **id-swap** (giữ tenant A, đổi ID) | `/orgs/{A}/tasks/{taskB}` | `404`: task không tồn tại *trong Org A* |

In [5]:
call("mock-a-owner", "GET",    f"/orgs/{ORG_B}/tasks/{task_b['id']}")
call("mock-a-owner", "GET",    f"/orgs/{ORG_A}/tasks/{task_b['id']}")
call("mock-a-owner", "PATCH",  f"/orgs/{ORG_A}/tasks/{task_b['id']}", {"title": "HACKED"})
call("mock-a-owner", "DELETE", f"/orgs/{ORG_A}/tasks/{task_b['id']}")
call(None,           "GET",    f"/orgs/{ORG_B}/projects")   # không token → 401
print("\nTask Org B sau tấn công:", call("mock-b-owner", "GET", f"/orgs/{ORG_B}/tasks/{task_b['id']}", show=False)["title"])

GET    /orgs/0246d1c9-6b70-581e-a854-8b67cbc9a65f/tasks/d397b784-3f53-533a-9e → 403  {"error": "FORBIDDEN", "message": "Không phải thành viên tổ chức này", "requestId": "req-dad4933ecf86"}
GET    /orgs/839467b6-9957-525f-9673-cf9a193b7635/tasks/d397b784-3f53-533a-9e → 404  {"error": "NOT_FOUND", "message": "Task không tồn tại", "requestId": "req-a284a8cf4a4c"}
PATCH  /orgs/839467b6-9957-525f-9673-cf9a193b7635/tasks/d397b784-3f53-533a-9e → 404  {"error": "NOT_FOUND", "message": "Task không tồn tại", "requestId": "req-a56b175ec55d"}
DELETE /orgs/839467b6-9957-525f-9673-cf9a193b7635/tasks/d397b784-3f53-533a-9e → 404  {"error": "NOT_FOUND", "message": "Task không tồn tại", "requestId": "req-88bd46adc0a7"}
GET    /orgs/0246d1c9-6b70-581e-a854-8b67cbc9a65f/projects                    → 401  {"error": "UNAUTHORIZED", "message": "Thiếu Bearer token", "requestId": "req-746dec0762e5"}

Task Org B sau tấn công: [B] Task 1


### B4. Guest chỉ được đọc: mọi thao tác ghi bị **API** chặn 403 (không chỉ ẩn nút trên UI)

In [6]:
call("mock-a-guest", "GET",  f"/orgs/{ORG_A}/projects", show=False) and print("GET    projects (guest)  → 200 OK")
call("mock-a-guest", "POST", f"/orgs/{ORG_A}/projects", {"name": "guest-try"})
call("mock-a-guest", "POST", f"/orgs/{ORG_A}/exports", {});

GET    projects (guest)  → 200 OK
POST   /orgs/839467b6-9957-525f-9673-cf9a193b7635/projects                    → 403  {"error": "FORBIDDEN", "message": "Role GUEST không được phép thao tác này", "requestId": "req-8f1c078728f3"}
POST   /orgs/839467b6-9957-525f-9673-cf9a193b7635/exports                     → 403  {"error": "FORBIDDEN", "message": "Role GUEST không được phép thao tác này", "requestId": "req-c70d8257e6d7"}


### B5. Trùng tên project: khác tenant thì OK, cùng tenant thì `409`

In [7]:
call("mock-a-owner", "POST", f"/orgs/{ORG_A}/projects", {"name": "Cloud Project"})
call("mock-b-owner", "POST", f"/orgs/{ORG_B}/projects", {"name": "Cloud Project"})
call("mock-a-owner", "POST", f"/orgs/{ORG_A}/projects", {"name": "Cloud Project"});  # trùng trong Org A

POST   /orgs/839467b6-9957-525f-9673-cf9a193b7635/projects                    → 201  {"id": "76efc423-988b-4d7d-ae4e-f5fc6b6d7777", "tenantId": "839467b6-9957-525f-9673-cf9a193b7635", "name": "Cloud Project", "description": "", "createdAt": "202…
POST   /orgs/0246d1c9-6b70-581e-a854-8b67cbc9a65f/projects                    → 201  {"id": "0385d7af-0468-473d-9775-ccb456bf9c2d", "tenantId": "0246d1c9-6b70-581e-a854-8b67cbc9a65f", "name": "Cloud Project", "description": "", "createdAt": "202…
POST   /orgs/839467b6-9957-525f-9673-cf9a193b7635/projects                    → 409  {"error": "CONFLICT", "message": "Tên project đã tồn tại trong tổ chức", "requestId": "req-26e7b1be7025"}


### B6. Thu hồi thành viên có hiệu lực ngay
User X (`mock-a-member3`) đang là MEMBER Org A → Owner xóa → **cùng token** gọi lại → `403`.

In [8]:
x = call("mock-a-member3", "GET", "/me", show=False)
call("mock-a-member3", "GET", f"/orgs/{ORG_A}/projects", show=False) and print("Trước khi thu hồi: X đọc được Org A → 200")
call("mock-a-owner",   "DELETE", f"/orgs/{ORG_A}/members/{x['userId']}")
call("mock-a-member3", "GET",    f"/orgs/{ORG_A}/projects");

Trước khi thu hồi: X đọc được Org A → 200
DELETE /orgs/839467b6-9957-525f-9673-cf9a193b7635/members/dda44448-97c1-595e- → 204  null
GET    /orgs/839467b6-9957-525f-9673-cf9a193b7635/projects                    → 403  {"error": "FORBIDDEN", "message": "Không phải thành viên tổ chức này", "requestId": "req-974100671534"}


### B7. Luồng Export: `202` → poll → `COMPLETED` → link SAS (đã che chữ ký khi in)

In [9]:
job = call("mock-a-owner", "POST", f"/orgs/{ORG_A}/exports", {})
call("mock-a-owner", "GET", f"/orgs/{ORG_A}/exports/{job['jobId']}/download")   # chưa xong → 409
time.sleep(2.5)
call("mock-a-owner", "GET", f"/orgs/{ORG_A}/exports/{job['jobId']}")
link = call("mock-a-owner", "GET", f"/orgs/{ORG_A}/exports/{job['jobId']}/download")
csv_head = requests.get(link["url"]).text.splitlines()[:3]
print("\nCSV (3 dòng đầu):", *csv_head, sep="\n  ")
print("SAS bị sửa chữ ký →", requests.get(link["url"].split("sig=")[0] + "sig=forged").status_code)
call("mock-a-owner", "GET", f"/orgs/{ORG_B}/exports/{job['jobId']}");   # Org A hỏi job qua path Org B → 403

POST   /orgs/839467b6-9957-525f-9673-cf9a193b7635/exports                     → 202  {"jobId": "9384ab49-2080-4e6d-ae22-c7f71447a81e", "status": "QUEUED"}
GET    /orgs/839467b6-9957-525f-9673-cf9a193b7635/exports/9384ab49-2080-4e6d- → 409  {"error": "NOT_READY", "message": "Export chưa hoàn tất", "requestId": "req-167498be2e0c"}


GET    /orgs/839467b6-9957-525f-9673-cf9a193b7635/exports/9384ab49-2080-4e6d- → 200  {"jobId": "9384ab49-2080-4e6d-ae22-c7f71447a81e", "tenantId": "839467b6-9957-525f-9673-cf9a193b7635", "projectId": null, "status": "COMPLETED", "requestedBy": "…
GET    /orgs/839467b6-9957-525f-9673-cf9a193b7635/exports/9384ab49-2080-4e6d- → 200  {"url": "http://127.0.0.1:53466/mock-blob/exports/839467b6-9957-525f-9673-cf9a193b7635/9384ab49-2080-4e6d-ae22-c7f71447a81e.csv?se=***REDACTED***&sp=***REDACTED…

CSV (3 dòng đầu):
  ﻿task_id,project_id,title,status,assignee_user_id,updated_at
  2cdafc8f-fa5c-5719-9da3-660b72c8b804,13ce0fb9-ccad-504c-adec-5f07cc974f32,[A] Task 1,TODO,a57921f4-f46a-5587-b205-f75550bf2d8c,2026-11-16T04:27:42+00:00
  a0632a0b-9030-5c29-a976-929215452aad,13ce0fb9-ccad-504c-adec-5f07cc974f32,[A] Task 2,IN_PROGRESS,f883003d-1721-5b67-9831-2ee8b6b3a9c0,2026-11-16T04:27:42+00:00
SAS bị sửa chữ ký → 403
GET    /orgs/0246d1c9-6b70-581e-a854-8b67cbc9a65f/exports/9384ab49-2080-4e6d- → 403

In [10]:
server.terminate(); server.wait(timeout=5)
print("Đã tắt mock API")

Đã tắt mock API


---
## C. ❓ Điểm cần TV2 xác nhận

| # | Câu hỏi | Đề xuất của TV4 | Ảnh hưởng nếu khác |
|---|---|---|---|
| 1 | Tenant trong path (`/orgs/{tenantId}/...`) hay header (`X-Tenant-Id`)? | Path | Sửa `api_paths.py` + `client.ts` |
| 2 | Truy cập tài nguyên tenant khác qua path tenant của mình trả `404` hay `403`? | `404` | Test 1 chấp nhận cả hai, miễn **không 2xx** và body không chứa dữ liệu Org B |
| 3 | Guest có được gọi `GET .../download` không? | Không (Guest chỉ xem) | Sửa điều kiện nút trên UI |
| 4 | Mời thành viên: thêm ngay hay tạo lời mời chờ chấp nhận? | Thêm ngay | Test 2 cần một cách **thêm lại** User X để chạy lặp lại |
| 6 | Tên trạng thái export: `QUEUED` hay `PENDING` (TV3 đang dùng)? Có `FAILED` không? | Theo TV3: `PENDING` + thêm `FAILED` cho lỗi không retry được | Frontend đã chịu được cả hai; chỉ cần sửa mock + tài liệu |
| 5 | Tên cột CSV & tên header request ID | `task_id, project_id, title, status, assignee_user_id, updated_at` · `X-Request-Id` | Sửa mock + tài liệu |